# EDA Figures

Plot all figures here.

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv('../data/processed/ca-obesity-living-cost.csv')

FIGURES_DIR = Path('../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

def aligned_histogram_edges(values, bin_width):
    start = np.floor(values.min() / bin_width) * bin_width
    stop = np.ceil(values.max() / bin_width) * bin_width
    return np.arange(start, stop + bin_width, bin_width)

## Individual Feature Plots

1. Boxplot
2. Histogram
3. Violin plot
4. ECDF plot

In [5]:
FEATURES = {
    'OBESITY_AdjPrev': 'Age-adjusted obesity prevalence (%)',
    'Living_Wage_Hourly_USD': 'Living wage (hourly USD)',
}

HISTOGRAM_BIN_WIDTHS = {
    'OBESITY_AdjPrev': 2,
    'Living_Wage_Hourly_USD': 1,
}

for feature, label in FEATURES.items():
    values = df[feature].dropna()
    FILE_PREFIX = feature.lower()

    # Boxplot
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.boxplot(values, orientation="horizontal", patch_artist=True, boxprops={'facecolor': '#4C78A8'})
    ax.set_title(f'Boxplot of {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{FILE_PREFIX}_boxplot.png', dpi=300)
    plt.close(fig)

    # Histogram
    fig, ax = plt.subplots(figsize=(8, 4.5))
    bin_edges = aligned_histogram_edges(values, HISTOGRAM_BIN_WIDTHS[feature])
    ax.hist(values, bins=bin_edges, color='#F58518', edgecolor='white')
    ax.set_xticks(bin_edges)
    ax.set_xlim(bin_edges[0], bin_edges[-1])
    ax.grid(axis='x', which='major')
    ax.set_title(f'Histogram of {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{FILE_PREFIX}_histogram.png', dpi=300)
    plt.close(fig)

    # Violin plot
    fig, ax = plt.subplots(figsize=(8, 4.5))
    parts = ax.violinplot(values, orientation="horizontal", showmedians=True, showextrema=True)
    for body in parts['bodies']:
        body.set_facecolor('#54A24B')
        body.set_edgecolor('#2F5F2B')
        body.set_alpha(0.75)
    ax.set_title(f'Violin plot of {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{FILE_PREFIX}_violin.png', dpi=300)
    plt.close(fig)

    # ECDF plot
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sorted_values = np.sort(values)
    cumulative = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
    ax.step(sorted_values, cumulative, where='post', color='#B279A2')
    ax.set_title(f'ECDF of {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('Cumulative proportion')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f'{FILE_PREFIX}_ecdf.png', dpi=300)
    plt.close(fig)


print("Generated figures:\n")
print("\n".join(sorted("\t"+path.name for path in FIGURES_DIR.glob('*.png'))))

Generated figures:

	living_wage_hourly_usd_boxplot.png
	living_wage_hourly_usd_ecdf.png
	living_wage_hourly_usd_histogram.png
	living_wage_hourly_usd_violin.png
	living_wage_obesity_correlation.png
	obesity_adjprev_boxplot.png
	obesity_adjprev_ecdf.png
	obesity_adjprev_histogram.png
	obesity_adjprev_violin.png


## Obesity vs. Living Wage

In [6]:
correlation_data = df[['Living_Wage_Hourly_USD', 'OBESITY_AdjPrev']]

# series
wage = correlation_data['Living_Wage_Hourly_USD']
obesity = correlation_data['OBESITY_AdjPrev']

# compute PCC
pcc = wage.corr(obesity, method='pearson')

# compute linear regression line
slope, intercept = np.polyfit(wage, obesity, 1)
line_x = np.linspace(wage.min(), wage.max(), 100)
line_y = slope * line_x + intercept

# create scatter plot with regression line and PCC
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(wage, obesity, color='#4C78A8', alpha=0.85, edgecolor='white', linewidth=0.6)
ax.plot(line_x, line_y, color='#E45756', linewidth=2)
ax.set_title('Correlation Between Living Wage and Obesity Prevalence')
ax.set_xlabel('Living wage (hourly USD)')
ax.set_ylabel('Age-adjusted obesity prevalence (%)')
ax.text(0.02, 0.95, f'PCC = {pcc:.3f}', transform=ax.transAxes, va='top')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'living_wage_obesity_correlation.png', dpi=300)
plt.close(fig)

print(f'PCC: {pcc:.3f}')

PCC: -0.781
